# Scrap Serenity (@aleabitoreddit)

X(Twitter) 트윗 수집 → `analysis_Serenity.db`

- **post**: 일반 게시물
- **reply**: 답글
- **subscriber**: 구독자 전용 (`exclusivityInfo` 기반 감지)

## 1. Configuration

쿠키와 설정값을 여기서 수정하세요.

In [1]:
# ── Cookie ────────────────────────────────────────────────────────────────────
# X(Twitter) 쿠키 — 만료 시 브라우저에서 새 값 복사
COOKIES = {
    "auth_token": "0732cb0b867deebea6ad8ca223fffa92a82c51f8",
    "ct0": "b0d3fcc9aa34f7f41736d0856a23baa3758536c0be1be45de601877529e58ac9a7cf238ec42432f170eb88c641aa10db1f9095875d48587c84998df013ce9edda7a75d44891dfb8676fbb2367032d04d",
    "twid": "u%3D1818125492823986176"
}

# ── Target ────────────────────────────────────────────────────────────────────
TARGET_USER = "aleabitoreddit"

# ── Options ───────────────────────────────────────────────────────────────────
TWEET_COUNT = None   # None = 전체, 숫자 = 최대 N개
DEBUG = False        # True = raw tweet._data JSON 덤프

# ── Paths (수정 불필요) ───────────────────────────────────────────────────────
import os
DB_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')),
    '..', '..', '..', '.claude', 'skills', 'MarketData', 'Personas', 'Serenity')

# 노트북 위치 기준 상대경로 → 절대경로로 고정
_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
DB_DIR = os.path.normpath(os.path.join(
    _notebook_dir, '..', 'Personas', 'Serenity'))
DB_PATH = os.path.join(DB_DIR, 'analysis_Serenity.db')

os.makedirs(DB_DIR, exist_ok=True)
print(f"DB path: {DB_PATH}")

DB path: /Users/seongjin/Documents/Seongjin_Claude/Dumok-of-WallStreet/.claude/skills/MarketData/Personas/Serenity/analysis_Serenity.db


## 2. Setup

In [2]:
import asyncio
import sqlite3
import json
import re
import tempfile
from datetime import datetime, timedelta, timezone
from twikit import Client

KST = timezone(timedelta(hours=9))
RATE_LIMIT_DELAY = 2
DUPLICATE_THRESHOLD = 10
MAX_RETRIES = 3


def to_kst(twitter_time_str):
    dt = datetime.strptime(twitter_time_str, "%a %b %d %H:%M:%S %z %Y")
    return dt.astimezone(KST).strftime("%Y-%m-%dT%H:%M:%S+09:00")


def extract_tickers(text):
    tickers = re.findall(r'\$([A-Za-z]+)', text)
    return list(dict.fromkeys([t.upper() for t in tickers]))


def get_full_content(tweet):
    note = (tweet._data
            .get('note_tweet', {})
            .get('note_tweet_results', {})
            .get('result', {}))
    if note and 'text' in note:
        return note['text']
    return tweet.full_text


def get_media_urls(tweet):
    urls = []
    if tweet.media:
        for m in tweet.media:
            if hasattr(m, 'media_url') and m.media_url:
                urls.append(m.media_url)
    return urls


def classify_tweet_type(tweet):
    if tweet._data.get('exclusivityInfo'):
        return 'subscriber'
    elif tweet.in_reply_to:
        return 'reply'
    else:
        return 'post'


def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS tweets (
            id TEXT PRIMARY KEY,
            user TEXT NOT NULL,
            type TEXT NOT NULL CHECK(type IN ('post', 'reply', 'subscriber')),
            created_at TEXT NOT NULL,
            content TEXT,
            tickers TEXT DEFAULT '[]',
            media TEXT DEFAULT '[]'
        )
    ''')
    conn.commit()
    return conn


def get_existing_ids(conn):
    cur = conn.execute('SELECT id FROM tweets')
    return set(row[0] for row in cur.fetchall())


def save_tweets(conn, tweets):
    saved = 0
    for tweet in tweets:
        content = get_full_content(tweet)
        tickers = json.dumps(extract_tickers(content), ensure_ascii=False)
        media = json.dumps(get_media_urls(tweet), ensure_ascii=False)
        tweet_type = classify_tweet_type(tweet)
        conn.execute(
            'INSERT OR IGNORE INTO tweets (id, user, type, created_at, content, tickers, media) VALUES (?,?,?,?,?,?,?)',
            (tweet.id, tweet.user.screen_name if tweet.user else TARGET_USER,
             tweet_type, to_kst(tweet.created_at), content, tickers, media))
        if conn.total_changes:
            saved += 1
    conn.commit()
    return saved


async def fetch_with_retry(coro_fn):
    for attempt in range(MAX_RETRIES):
        try:
            return await coro_fn()
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise
            wait = 2 ** (attempt + 1)
            print(f"  ⚠ {e} — retry in {wait}s...")
            await asyncio.sleep(wait)


print("Setup complete.")

Setup complete.


## 3. Scrape

In [3]:
async def run_scrape():
    conn = init_db()
    existing_ids = get_existing_ids(conn)
    print(f"DB: {DB_PATH}")
    print(f"Existing: {len(existing_ids)}")

    # 쿠키를 임시 파일로 저장하여 twikit에 전달
    cookie_file = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
    json.dump(COOKIES, cookie_file)
    cookie_file.close()

    client = Client('en-US')
    try:
        client.load_cookies(cookie_file.name)
    except Exception as e:
        print(f"✗ Cookie expired — update COOKIES in cell 1.\n  {e}")
        conn.close()
        return
    finally:
        os.unlink(cookie_file.name)

    user = await client.get_user_by_screen_name(TARGET_USER)
    print(f"\n=== {user.name} (@{user.screen_name}) ===")
    print(f"Followers: {user.followers_count:,} | Tweets: {user.statuses_count:,}")

    total_saved = 0
    type_counts = {'post': 0, 'reply': 0, 'subscriber': 0}

    for tweet_type in ['Tweets', 'Replies']:
        print(f"\nFetching {tweet_type}...")
        consecutive_dups = 0
        total_fetched = 0

        try:
            tweets = await fetch_with_retry(
                lambda tt=tweet_type: user.get_tweets(tt, count=20))
        except Exception as e:
            print(f"  ⚠ Failed: {e}")
            continue

        while tweets:
            page_tweets = []
            for t in tweets:
                sn = t.user.screen_name if t.user else None
                if sn and sn.lower() != TARGET_USER.lower():
                    continue

                total_fetched += 1

                if DEBUG:
                    print(f"  [DEBUG] {t.id}: exclusivityInfo={t._data.get('exclusivityInfo')}, in_reply_to={t.in_reply_to}")

                if t.id in existing_ids:
                    consecutive_dups += 1
                    if consecutive_dups >= DUPLICATE_THRESHOLD:
                        print(f"  {DUPLICATE_THRESHOLD} consecutive dups — stopping")
                        break
                else:
                    consecutive_dups = 0
                    tt = classify_tweet_type(t)
                    type_counts[tt] += 1
                    page_tweets.append(t)
                    existing_ids.add(t.id)
                    if TWEET_COUNT and (total_saved + len(page_tweets)) >= TWEET_COUNT:
                        break

            if page_tweets:
                saved = save_tweets(conn, page_tweets)
                total_saved += saved
                print(f"  Fetched {total_fetched}, saved {total_saved} total")

            if consecutive_dups >= DUPLICATE_THRESHOLD:
                break
            if TWEET_COUNT and total_saved >= TWEET_COUNT:
                break

            await asyncio.sleep(RATE_LIMIT_DELAY)
            try:
                tweets = await fetch_with_retry(lambda: tweets.next())
                if not tweets:
                    break
            except Exception as e:
                print(f"  ⚠ Pagination stopped: {e}")
                break

        if TWEET_COUNT and total_saved >= TWEET_COUNT:
            break

    print(f"\n{'='*40}")
    print(f"Saved this run: {total_saved}")
    print(f"  post={type_counts['post']}, reply={type_counts['reply']}, subscriber={type_counts['subscriber']}")

    # DB 통계
    cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
    stats = cur.fetchall()
    total = sum(c for _, c in stats)
    print(f"\nTotal in DB: {total}")
    for t, c in stats:
        print(f"  {t}: {c}")
    conn.close()

await run_scrape()

DB: /Users/seongjin/Documents/Seongjin_Claude/Dumok-of-WallStreet/.claude/skills/MarketData/Personas/Serenity/analysis_Serenity.db
Existing: 0

=== Serenity (@aleabitoreddit) ===
Followers: 110,071 | Tweets: 4,347

Fetching Tweets...
  Fetched 19, saved 19 total
  Fetched 37, saved 37 total
  Fetched 55, saved 55 total
  Fetched 74, saved 74 total
  Fetched 91, saved 91 total
  Fetched 106, saved 106 total
  Fetched 123, saved 123 total
  Fetched 139, saved 139 total
  Fetched 158, saved 158 total
  Fetched 174, saved 174 total
  Fetched 192, saved 192 total
  Fetched 210, saved 210 total
  Fetched 228, saved 228 total
  Fetched 247, saved 247 total
  Fetched 265, saved 265 total
  Fetched 281, saved 281 total
  Fetched 294, saved 294 total
  Fetched 313, saved 313 total
  Fetched 327, saved 327 total
  Fetched 345, saved 345 total
  Fetched 361, saved 361 total
  Fetched 381, saved 381 total
  Fetched 398, saved 398 total
  Fetched 417, saved 417 total
  Fetched 435, saved 435 total
 

## 4. Explore DB

In [4]:
# ── 최신 트윗 확인 ─────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    'SELECT id, type, created_at, substr(content,1,80), tickers FROM tweets ORDER BY created_at DESC LIMIT 10')
for row in cur:
    print(f"[{row[1]:10s}] {row[2]}  {row[3]}")
    if row[4] != '[]':
        print(f"             tickers: {row[4]}")
conn.close()

[post      ] 2026-03-18T13:40:34+09:00  The more I look into Sivers 
< $SIVE /  $SIVEF >

the more I wonder:

How is the
             tickers: ["SIVE", "SIVEF", "LWLG", "AXTI"]
[post      ] 2026-03-18T11:30:30+09:00  $AAOI looks very undervalued at $6.49B. 

If we model ASP and their newest capac
             tickers: ["AAOI", "AMZN", "MSFT", "LITE", "COHR"]
[post      ] 2026-03-18T10:59:32+09:00  $SIVE is the upstream laser supplier for CPO and Silicon Photonics.

They're the
             tickers: ["SIVE", "COHR", "LITE", "AMZN", "MSFT", "META", "GOOGL", "POET", "MRVL", "TPU", "MTSI"]
[post      ] 2026-03-18T08:58:00+09:00  The Photonics Supercycle is here.

$NVDA is spearheading the next leap into CPO 
             tickers: ["NVDA", "SOI", "SIVE"]
[post      ] 2026-03-17T23:34:50+09:00  I never gave up hope in my lobster companion $RPI. 

Glad to see it’s finally ge
             tickers: ["RPI"]
[post      ] 2026-03-17T21:38:10+09:00  The upcoming CPO / Silicon Photonics Bottleneck C

In [5]:
# ── 타입별 통계 ───────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
for t, c in cur:
    print(f"  {t}: {c}")
total = conn.execute('SELECT COUNT(*) FROM tweets').fetchone()[0]
print(f"  total: {total}")
conn.close()

  post: 662
  reply: 1
  subscriber: 104
  total: 767


In [6]:
# ── 구독자 전용 트윗 확인 ─────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT created_at, substr(content,1,100), tickers FROM tweets WHERE type='subscriber' ORDER BY created_at DESC LIMIT 10")
for row in cur:
    print(f"{row[0]}  {row[1]}")
    if row[2] != '[]':
        print(f"  tickers: {row[2]}")
    print()
conn.close()

2026-03-17T13:32:33+09:00  Cycles. Are. Extremely Important. When timing markets. 

Here's what I've learned so far over the ye
  tickers: ["MSTR", "RKLB", "CPSH", "AEHR", "POET", "MRVL", "SIVE", "AXTI", "AAOI", "MSFT", "AMZN", "SNDK", "LITE", "COHR", "NVDA", "ZOOM", "HOOD", "AMKR", "NBIS"]

2026-03-17T07:41:40+09:00  Still debating Nokia after Infinera since they now have a 6n InP fab for 2026 ramp.

I personally ha
  tickers: ["COHR", "TSEM"]

2026-03-16T17:42:27+09:00  New position TLDR:

$SIVE at $140M MC. 

Laser supplier for upcoming CPO/silicon photonics companies
  tickers: ["SIVE", "LITE", "POET", "AYAR", "COHR", "MRVL"]

2026-03-15T20:55:11+09:00  Okay so basically research list:

$MP, $NEO, $ATI $CRS are anchors for robotic supply chains. 

1. M
  tickers: ["MP", "NEO", "ATI", "CRS", "UUUU", "ALOY", "USAR", "LYSDY", "ILU", "ARU", "FCX", "NB", "MTRN", "LGO", "BMM", "VNP", "TECK", "ALB", "EAF", "ALTM", "SYR", "AW"]

2026-03-15T19:32:20+09:00  Since there's no US companies on r

In [7]:
# ── 특정 티커 검색 ────────────────────────────────────────────────────────
SEARCH_TICKER = "AXTI"  # ← 변경하여 검색

conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT type, created_at, substr(content,1,100) FROM tweets WHERE tickers LIKE ? ORDER BY created_at DESC",
    (f'%"{SEARCH_TICKER}"%',))
rows = cur.fetchall()
print(f"${SEARCH_TICKER} mentions: {len(rows)}")
for row in rows[:10]:
    print(f"  [{row[0]}] {row[1]}  {row[2]}")
conn.close()

$AXTI mentions: 107
  [post] 2026-03-18T13:40:34+09:00  The more I look into Sivers 
< $SIVE /  $SIVEF >

the more I wonder:

How is the laser company for t
  [post] 2026-03-17T21:38:10+09:00  The upcoming CPO / Silicon Photonics Bottleneck Cheat Sheet:

$SIVE, Sumitomo, $LITE, $COHR, $AVGO, 
  [post] 2026-03-17T15:27:57+09:00  Woah, Macronix (TPE: 2337) is out here casually outperforming $AXTI? 

201.04% vs. 188% Year to Date
  [subscriber] 2026-03-17T13:32:33+09:00  Cycles. Are. Extremely Important. When timing markets. 

Here's what I've learned so far over the ye
  [post] 2026-03-17T10:43:40+09:00  Normally don’t respond to trolls, but the hypocrisy on this platform is pretty impressive. 

Random 
  [post] 2026-03-16T20:10:51+09:00  $AXTI is now above $50+.

I was the first to identify it as the next major bottleneck for the AI bui
  [post] 2026-03-16T17:54:18+09:00  I’m long $SIVE at $140M. 

I believe this is the next $LITE that markets and institutions missed.

$
  [post] 2026-0